In [3]:
## Bibliotecas principais do Módulo 4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [4]:
# Estrutura padrão do projeto
RAIZ = Path.cwd().parent.resolve()

PASTAS = ["dados_brutos", "dados_tratados", "docs",
 "notebooks", "sql", "dashboards", "resultados",
 "relatorios", "apresentacao", "logs"]

for pasta in PASTAS:
 (RAIZ / pasta).mkdir(
 parents=True, exist_ok=True)

print("Pastas verificadas/criadas:")
for pasta in PASTAS:
 print("-", RAIZ / pasta)

Pastas verificadas/criadas:
- /workspaces/FAP-2026-AnaliseDados/dados_brutos
- /workspaces/FAP-2026-AnaliseDados/dados_tratados
- /workspaces/FAP-2026-AnaliseDados/docs
- /workspaces/FAP-2026-AnaliseDados/notebooks
- /workspaces/FAP-2026-AnaliseDados/sql
- /workspaces/FAP-2026-AnaliseDados/dashboards
- /workspaces/FAP-2026-AnaliseDados/resultados
- /workspaces/FAP-2026-AnaliseDados/relatorios
- /workspaces/FAP-2026-AnaliseDados/apresentacao
- /workspaces/FAP-2026-AnaliseDados/logs


In [5]:
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

ARQUIVO_BRUTO = RAIZ / "dados_brutos" / "acidentes2025.csv"
ARQUIVO_BASE_ANALITICA = RAIZ / "dados_tratados" / "base_analitica_prf_2025.csv"
ARQUIVO_BASE_MODELAVEL = RAIZ / "dados_tratados" / "base_modelavel_prf_2025.csv"
ARQUIVO_DICIONARIO = RAIZ / "dados_tratados" / "dicionario_variaveis_modulo4.csv"
ARQUIVO_DECISOES = RAIZ / "logs" / "decisoes_tratamento_modulo4.md"
ARQUIVO_README = RAIZ / "docs" / "README.md"

SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [6]:
from pathlib import Path

print("Diretório atual:", Path.cwd())
print("Arquivo existe?", Path(ARQUIVO_BRUTO).exists())
print("Caminho:", Path(ARQUIVO_BRUTO).resolve())

Diretório atual: /workspaces/FAP-2026-AnaliseDados/notebook
Arquivo existe? False
Caminho: /workspaces/FAP-2026-AnaliseDados/notebook/dados_brutos/acidentes2025.csv


In [7]:
from pathlib import Path

ARQUIVO_BRUTO = Path("../dados_brutos/acidentes2025.csv")

print(ARQUIVO_BRUTO.resolve())
print("Existe?", ARQUIVO_BRUTO.exists())

/workspaces/FAP-2026-AnaliseDados/dados_brutos/acidentes2025.csv
Existe? True


In [8]:
def ler_csv_prf(caminho, sep=";",
 encodings=("latin1","utf-8","utf-8-sig")):
 ultimo_erro = None
 for enc in encodings:
    try:
        print(f"Tentando encoding={enc}...")
        return pd.read_csv(
        caminho, sep=sep,
        encoding=enc, low_memory=False)
    except Exception as erro:
        ultimo_erro = erro
        print(f"Falhou com {enc}: {erro}")
        raise ultimo_erro

df = ler_csv_prf(ARQUIVO_BRUTO, sep=SEPARADOR)
df.head()

Tentando encoding=latin1...


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [9]:
def normalizar_nome_coluna(nome):
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize(
        "NFKD", nome
    ).encode("ascii","ignore").decode("utf-8")
    nome = nome.replace(" ","_"
        ).replace("-","_").replace("/","_")
    while "__" in nome:
        nome = nome.replace("__","_")
    return nome.strip("_")

df.columns = [normalizar_nome_coluna(c)
              for c in df.columns]
renomear = {
  "condicao_meteorologica":
  "condicao_metereologica"}
df = df.rename(columns={
    k:v for k,v in renomear.items()
    if k in df.columns})

df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [10]:
colunas_esperadas = ["data_inversa","dia_semana","horario",
 "uf","br","municipio","causa_acidente",
 "tipo_acidente","classificacao_acidente",
 "fase_dia","condicao_metereologica",
 "tipo_pista","tracado_via","uso_solo",
 "pessoas","mortos","feridos_leves",
 "feridos_graves","feridos","veiculos"]

faltantes = [c for c in colunas_esperadas
 if c not in df.columns]
print("Colunas faltantes:", faltantes)

if faltantes:
 print("Atenção: ajuste nomes ou confirme o dicionário da PRF.")

Colunas faltantes: []


In [11]:
# Tipos de dados e memória utilizada
df.info(memory_usage="deep")

resumo_tipos = (
 df.dtypes.astype(str)
 .value_counts()
 .rename_axis("tipo")
 .reset_index(name="qtd_colunas")
)
display(resumo_tipos)

<class 'pandas.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id                      72529 non-null  int64
 1   data_inversa            72529 non-null  str  
 2   dia_semana              72529 non-null  str  
 3   horario                 72529 non-null  str  
 4   uf                      72529 non-null  str  
 5   br                      72529 non-null  int64
 6   km                      72529 non-null  str  
 7   municipio               72529 non-null  str  
 8   causa_acidente          72529 non-null  str  
 9   tipo_acidente           72529 non-null  str  
 10  classificacao_acidente  72528 non-null  str  
 11  fase_dia                72529 non-null  str  
 12  sentido_via             72529 non-null  str  
 13  condicao_metereologica  72529 non-null  str  
 14  tipo_pista              72529 non-null  str  
 15  tracado_via             72529 

,tipo,qtd_colunas
0,str,20
1,int64,10


In [12]:
# Diagnóstico de valores ausentes
nulos = pd.DataFrame({
    "qtd_nulos": df.isna().sum(),
    "perc_nulos": df.isna().mean() * 100
}).sort_values(
    "perc_nulos", ascending=False)

display(nulos[nulos["qtd_nulos"] > 0])

,qtd_nulos,perc_nulos
uop,38,0.052393
delegacia,22,0.030333
regional,2,0.002758
classificacao_acidente,1,0.001379


In [13]:
# Diagnóstico e remoção de duplicidades
qtd_duplicadas = df.duplicated().sum()
print("Duplicidades exatas:", qtd_duplicadas)

if qtd_duplicadas > 0:
    df = df.drop_duplicates().copy()
    print("Duplicidades removidas.")
    print("Nova dimensão:", df.shape)

df.shape

Duplicidades exatas: 0


(72529, 30)

In [14]:
# Cardinalidade das variáveis categóricas
categoricas = df.select_dtypes(
 include="object").columns

cardinalidade = (
 df[categoricas]
 .nunique(dropna=True)
 .sort_values(ascending=False)
 .reset_index()
)
cardinalidade.columns = [
 "variavel","qtd_categorias"]
display(cardinalidade.head(30))

/tmp/ipykernel_6120/1504401738.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(


,variavel,qtd_categorias
0,latitude,69294
1,longitude,69237
2,km,7655
3,municipio,1844
4,horario,1412
5,tracado_via,605
6,uop,395
7,data_inversa,365
8,delegacia,153
9,causa_acidente,69


In [15]:
# Autor: Danilo Farias
# Copiloto: ChatGPT
# Conversão segura de valores numéricos com vírgula decimal
# Ex.: "546,2" -> 546.2

colunas_para_converter = ["br","km","pessoas","mortos","feridos",
 "feridos_leves","feridos_graves",
 "ilesos","ignorados","veiculos"]

for coluna in colunas_para_converter:
    if coluna in df.columns:
        valores_originais = df[coluna]

        valores_texto = (
            valores_originais.astype("string")
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

        valores_convertidos = pd.to_numeric(
            valores_texto,
            errors="coerce"
        )

        # Impede perda silenciosa de valores inválidos
        falhas = (
            valores_originais.notna()
            & valores_convertidos.isna()
        )

        if falhas.any():
            exemplos = valores_originais[falhas].head().tolist()
            raise ValueError(
                f"Falha na conversão da coluna '{coluna}'. "
                f"Exemplos: {exemplos}"
            )

        df[coluna] = valores_convertidos

display(df[colunas_para_converter].dtypes)

br                  Int64
km                Float64
pessoas             Int64
mortos              Int64
feridos             Int64
feridos_leves       Int64
feridos_graves      Int64
ilesos              Int64
ignorados           Int64
veiculos            Int64
dtype: object

In [16]:
 # Autor: Danilo Farias
# Copiloto: ChatGPT
df["data_inversa"] = pd.to_datetime(
    df["data_inversa"],
    format="%d/%m/%Y",
    errors="coerce"
)

df["ano"] = df["data_inversa"].dt.year
df["mes"] = df["data_inversa"].dt.month
df["trimestre"] = df["data_inversa"].dt.quarter
df["dia_semana_num"] = df["data_inversa"].dt.dayofweek
df["fim_de_semana"] = df["dia_semana_num"].isin([5, 6]).astype(int)

df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,546.2,PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0


In [17]:
horario_limpo = df["horario"].astype(str).str.strip()
df["hora"] = pd.to_datetime(
 horario_limpo, format="%H:%M:%S",
 errors="coerce").dt.hour

def classificar_turno(hora):
    if pd.isna(hora): return "IGNORADO"
    if 0 <= hora <= 5: return "MADRUGADA"
    if 6 <= hora <= 11: return "MANHA"
    if 12 <= hora <= 17: return "TARDE"
    return "NOITE"

df["turno"] = df["hora"].apply(classificar_turno)


In [18]:
def criar_faixa_horaria(hora):
    if pd.isna(hora):
        return "IGNORADO"
    inicio = int(hora // 3) * 3
    fim = inicio + 2
    return f"{inicio:02d}h-{fim:02d}h"

df["faixa_horaria"] = df["hora"].apply(criar_faixa_horaria)

display(df["faixa_horaria"].value_counts(dropna=False).sort_index())

faixa_horaria
00h-02h     3959
03h-05h     4948
06h-08h    11517
09h-11h     9342
12h-14h     9678
15h-17h    12624
18h-20h    13473
21h-23h     6988
Name: count, dtype: int64

In [19]:
# Autor: Danilo Farias
# Copiloto: ChatGPT
# Limpeza das colunas textuais
colunas_string = df.select_dtypes(
    include=["object", "string"]
).columns

for coluna in colunas_string:
    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({
            "": pd.NA,
            "NAN": pd.NA,
            "NULL": pd.NA
        })
    )

display(df[colunas_string].head())

,dia_semana,horario,uf,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,latitude,longitude,regional,delegacia,uop,turno,faixa_horaria
0,QUARTA-FEIRA,06:20:00,SP,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,RETA;DECLIVE,SIM,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,MANHA,06H-08H
1,QUARTA-FEIRA,07:50:00,CE,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,<NA>,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,RETA,NÃO,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,MANHA,06H-08H
2,QUARTA-FEIRA,08:45:00,PR,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,RETA;ACLIVE,SIM,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,MANHA,06H-08H
3,QUARTA-FEIRA,11:00:00,PR,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,RETA,NÃO,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,MANHA,09H-11H
4,QUARTA-FEIRA,09:30:00,MG,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,CURVA;DECLIVE,NÃO,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,MANHA,09H-11H


In [20]:
categoricas_importantes = ["uf","municipio","causa_acidente",
 "tipo_acidente","fase_dia",
 "condicao_metereologica","tipo_pista",
 "tracado_via","uso_solo",
 "classificacao_acidente","dia_semana"]

for coluna in categoricas_importantes:
    if coluna in df.columns:
        df[coluna] = df[coluna].fillna("IGNORADO")

print(df[categoricas_importantes]
 .isna().sum()
 .sort_values(ascending=False))

uf                        0
municipio                 0
causa_acidente            0
tipo_acidente             0
fase_dia                  0
condicao_metereologica    0
tipo_pista                0
tracado_via               0
uso_solo                  0
classificacao_acidente    0
dia_semana                0
dtype: int64


In [21]:
contagens_vitimas = ["mortos","feridos","feridos_leves",
 "feridos_graves","pessoas","veiculos"]

for coluna in contagens_vitimas:
    if coluna in df.columns:
        df[coluna] = df[coluna].fillna(0)

print(df[[c for c in contagens_vitimas
 if c in df.columns]].isna().sum())

mortos            0
feridos           0
feridos_leves     0
feridos_graves    0
pessoas           0
veiculos          0
dtype: int64


In [22]:
df.head(5)

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana,hora,turno,faixa_horaria
0,652493,2025-01-01,QUARTA-FEIRA,06:20:00,SP,116,225.0,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,RETA;DECLIVE,SIM,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0,6,MANHA,06H-08H
1,652519,2025-01-01,QUARTA-FEIRA,07:50:00,CE,116,546.2,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,IGNORADO,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,RETA,NÃO,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0,7,MANHA,06H-08H
2,652522,2025-01-01,QUARTA-FEIRA,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,RETA;ACLIVE,SIM,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0,8,MANHA,06H-08H
3,652544,2025-01-01,QUARTA-FEIRA,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,RETA,NÃO,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0,11,MANHA,09H-11H
4,652549,2025-01-01,QUARTA-FEIRA,09:30:00,MG,251,471.0,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,CURVA;DECLIVE,NÃO,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0,9,MANHA,09H-11H


In [23]:
df["acidente_fatal"] = np.where(
    df["mortos"] >= 1, 1, 0)

validacao_alvo = (
  df["acidente_fatal"]
  .value_counts(dropna=False)
  .rename_axis("acidente_fatal")
  .reset_index(name="qtd"))
validacao_alvo["perc"] = (
  validacao_alvo["qtd"] /
  validacao_alvo["qtd"].sum() * 100)
display(validacao_alvo)

,acidente_fatal,qtd,perc
0,0,67319,92.816666
1,1,5210,7.183334


In [24]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano,mes,trimestre,dia_semana_num,fim_de_semana,hora,turno,faixa_horaria,acidente_fatal
0,652493,2025-01-01,QUARTA-FEIRA,06:20:00,SP,116,225.0,GUARULHOS,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,TOMBAMENTO,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CÉU CLARO,MÚLTIPLA,RETA;DECLIVE,SIM,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,2025,1,1,2,0,6,MANHA,06H-08H,0
1,652519,2025-01-01,QUARTA-FEIRA,07:50:00,CE,116,546.2,PENAFORTE,PISTA ESBURACADA,COLISÃO FRONTAL,IGNORADO,PLENO DIA,CRESCENTE,CÉU CLARO,SIMPLES,RETA,NÃO,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,1,1,2,0,7,MANHA,06H-08H,1
2,652522,2025-01-01,QUARTA-FEIRA,08:45:00,PR,369,88.2,CORNELIO PROCOPIO,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO TRASEIRA,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,SOL,DUPLA,RETA;ACLIVE,SIM,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,2025,1,1,2,0,8,MANHA,06H-08H,0
3,652544,2025-01-01,QUARTA-FEIRA,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,SAÍDA DE LEITO CARROÇÁVEL,COM VÍTIMAS FERIDAS,PLENO DIA,CRESCENTE,CÉU CLARO,DUPLA,RETA,NÃO,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,2025,1,1,2,0,11,MANHA,09H-11H,0
4,652549,2025-01-01,QUARTA-FEIRA,09:30:00,MG,251,471.0,FRANCISCO SA,VELOCIDADE INCOMPATÍVEL,COLISÃO FRONTAL,COM VÍTIMAS FERIDAS,PLENO DIA,DECRESCENTE,CHUVA,SIMPLES,CURVA;DECLIVE,NÃO,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,2025,1,1,2,0,9,MANHA,09H-11H,0


In [25]:
def ajuste_classificacao_acidente(mortos, feridos):
    if mortos > 0:
        return "COM VÍTIMAS FATAIS"
    elif feridos > 0:
        return "COM VÍTIMAS FERIDAS"
    else:
        return "SEM VÍTIMAS"

df["classificacao_acidente"] = df.apply(lambda row: ajuste_classificacao_acidente(row["mortos"], row["feridos"]), axis=1)

display(df["classificacao_acidente"].value_counts(dropna=False).sort_index())

classificacao_acidente
COM VÍTIMAS FATAIS      5210
COM VÍTIMAS FERIDAS    56181
SEM VÍTIMAS            11138
Name: count, dtype: int64

In [26]:
df["tracado_via"] = df["tracado_via"].str.split(";")

display(df["tracado_via"].value_counts(dropna=False).sort_index())

tracado_via
[ACLIVE]                                                           745
[ACLIVE, CURVA]                                                    696
[ACLIVE, CURVA, EM OBRAS]                                            5
[ACLIVE, CURVA, INTERSEÇÃO DE VIAS]                                 11
[ACLIVE, CURVA, INTERSEÇÃO DE VIAS, EM OBRAS]                        1
                                                                  ... 
[VIADUTO, RETORNO REGULAMENTADO, INTERSEÇÃO DE VIAS, ROTATÓRIA]      1
[VIADUTO, RETORNO REGULAMENTADO, RETA]                               1
[VIADUTO, ROTATÓRIA]                                                13
[VIADUTO, ROTATÓRIA, INTERSEÇÃO DE VIAS]                             1
[VIADUTO, ROTATÓRIA, RETORNO REGULAMENTADO]                          1
Name: count, Length: 605, dtype: int64

In [27]:
# Autor: Danilo Farias
# Copiloto: ChatGPT
# Criação dos Indicadores de Gravidade do Acidente
df["total_vitimas"] = df["mortos"] + df["feridos"]

df["acidente_grave"] = np.where(
    (df["mortos"] >= 1) | (df["feridos_graves"] >= 1), 1, 0)

df["indice_gravidade"] = df["mortos"] * 3 + df["feridos_graves"] * 2 + df["feridos_leves"] * 1

In [28]:
def formatar_br(valor):
    if pd.isna(valor) or valor == 0:
        return "BR-IGNORADA"
    return f"BR-{int(valor):03d}"

df["br_formatada"] = df["br"].apply(formatar_br)

df["chave_localidade"] = (
        df["uf"].astype(str) + "_" +
        df["municipio"].astype(str) + "_" +
        df["br_formatada"].astype(str)
)

display(df[["uf","municipio","br",
 "br_formatada","chave_localidade"]].head())

,uf,municipio,br,br_formatada,chave_localidade
0,SP,GUARULHOS,116,BR-116,SP_GUARULHOS_BR-116
1,CE,PENAFORTE,116,BR-116,CE_PENAFORTE_BR-116
2,PR,CORNELIO PROCOPIO,369,BR-369,PR_CORNELIO PROCOPIO_BR-369
3,PR,CAMPINA GRANDE DO SUL,116,BR-116,PR_CAMPINA GRANDE DO SUL_BR-116
4,MG,FRANCISCO SA,251,BR-251,MG_FRANCISCO SA_BR-251


In [29]:
checagens = {
  "linhas": len(df),
  "colunas": df.shape[1],
  "acidentes_fatais": int(df["acidente_fatal"].sum()),
  "taxa_fatalidade": float(df["acidente_fatal"].mean()),
  "total_mortos": int(df["mortos"].sum()),
}
checagens

{'linhas': 72529,
 'colunas': 44,
 'acidentes_fatais': 5210,
 'taxa_fatalidade': 0.07183333563126473,
 'total_mortos': 6043}

In [30]:
def ranking_categoria(base, coluna, n=10):
 return (
 base[coluna]
 .value_counts(dropna=False)
 .head(n)
 .rename_axis(coluna)
 .reset_index(name="qtd")
 )

display(ranking_categoria(df, "causa_acidente", 10))
display(ranking_categoria(df, "tipo_acidente", 10))
display(ranking_categoria(df, "classificacao_acidente", 10))

,causa_acidente,qtd
0,AUSÊNCIA DE REAÇÃO DO CONDUTOR,11469
1,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,10799
2,ACESSAR A VIA SEM OBSERVAR A PRESENÇA DOS OUTR...,7097
3,CONDUTOR DEIXOU DE MANTER DISTÂNCIA DO VEÍCULO...,4413
4,VELOCIDADE INCOMPATÍVEL,4088
5,MANOBRA DE MUDANÇA DE FAIXA,4016
6,INGESTÃO DE ÁLCOOL PELO CONDUTOR,3685
7,DEMAIS FALHAS MECÂNICAS OU ELÉTRICAS,3385
8,TRANSITAR NA CONTRAMÃO,2475
9,CONDUTOR DORMINDO,2116


,tipo_acidente,qtd
0,COLISÃO TRASEIRA,14360
1,SAÍDA DE LEITO CARROÇÁVEL,10209
2,COLISÃO TRANSVERSAL,9306
3,COLISÃO LATERAL MESMO SENTIDO,7885
4,TOMBAMENTO,6351
5,COLISÃO COM OBJETO,5109
6,COLISÃO FRONTAL,4739
7,QUEDA DE OCUPANTE DE VEÍCULO,3450
8,ATROPELAMENTO DE PEDESTRE,3057
9,COLISÃO LATERAL SENTIDO OPOSTO,2152


,classificacao_acidente,qtd
0,COM VÍTIMAS FERIDAS,56181
1,SEM VÍTIMAS,11138
2,COM VÍTIMAS FATAIS,5210
